# 1. Tabular MLP Reranker with Loss and Negative-Sampling Ablation

The MLP scores each candidate from the frozen causal tabular features alone. Its
architecture is held fixed while the training objective varies:

- **BCE**: weighted binary cross-entropy on individual candidates
- **BPR**: pairwise loss that pushes a click above an unclicked candidate from the same
  impression, with uniform or popularity-weighted negative sampling
- **Hybrid**: $(1-w)\,\text{BCE} + w\,\text{BPR}$ for $w \in \{0.25, 0.50, 0.75\}$

Every trial uses the same training impressions, fitted preprocessor, seed, optimiser and
epoch budget. Validation nDCG@10 selects the final trial; only that trial is tested.

**Data limitation:** feature engineering already sampled the training negatives, so BPR
draws from the *retained* unclicked candidates. Validation and test use complete slates.

## 1.1 Load Packages

In [ ]:
import json
import random
import sys
from itertools import cycle
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [ ]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path("/Users/numkoos/Downloads/BT4222-Group-6-")
    DATA_ROOT = Path("/Users/numkoos/Downloads/v3-mindsmall")

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import compare_results, rank_logged_slate, score_and_evaluate

## 1.3 Load Ranking Features

Each split is stored as Parquet partitions with one row per (impression, candidate)
pair. These previews read only the first rows; later cells load what they need.

In [ ]:
def preview_split(split, n_rows=5):
    """Read the first rows of a split without loading every partition."""
    first_part = sorted((RUN / f"ranking_features_{split}").glob("*.parquet"))[0]
    return next(pq.ParquetFile(first_part).iter_batches(batch_size=n_rows)).to_pandas()


train_preview = preview_split("train")
validation_preview = preview_split("validation")
train_preview.head()

In [ ]:
validation_preview.head()

# 2. Experiment Configuration

`POPULARITY_ALPHAS` sets the BPR negative-sampling weight `(1 + prior exposures) ** alpha`:
0 is uniform and larger values favour popular articles. `HYBRID_WEIGHTS` is the BPR share
of the hybrid loss.

In [ ]:
SEED = 4222
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Using {DEVICE}")

TRAIN_IMPRESSIONS = 10_000      # sampled once and shared by every trial
NEGATIVES_PER_POSITIVE = 4      # unclicked candidates paired with each click for BPR

HIDDEN_DIMS = (256, 64)
DROPOUT = 0.20
EPOCHS = 5
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

POPULARITY_ALPHAS = (0.0, 0.5, 1.0)
HYBRID_WEIGHTS = (0.25, 0.50, 0.75)
SELECTION_METRIC = "ndcg_at_10"
OUTPUT_DIR = ARTIFACTS / "mlp_loss_ablation"

In [ ]:
# The frozen feature schema lists the causal tabular inputs shared by every reranker
schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
CATEGORICAL = [name for name in schema.get("categorical_columns", []) if name in FEATURES]
NUMERIC = [name for name in FEATURES if name not in CATEGORICAL]
print(f"{len(FEATURES)} tabular features: {len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical")

# 3. Prepare Training Data

## 3.1 Sample Training Impressions

Impressions (not individual rows) are sampled so that every click keeps the unclicked
candidates it was shown with.

In [ ]:
def sample_train_impressions(max_impressions, columns):
    """Sample whole training impressions, then load every retained candidate in them.

    Keeping impressions whole lets BPR pair each click with unclicked candidates from
    the same slate. Feature engineering already sampled training negatives, so these
    are the retained candidates rather than the complete displayed slate.
    """
    feature_files = sorted((RUN / "ranking_features_train").glob("*.parquet"))
    label_files = sorted((RUN / "labels_train").glob("*.parquet"))
    impression_ids = (
        pl.scan_parquet(label_files)
        .select("impression_uid").unique().sort("impression_uid")
        .collect()["impression_uid"].to_numpy()
    )
    rng = np.random.default_rng(SEED)
    chosen = rng.choice(impression_ids, size=min(max_impressions, len(impression_ids)), replace=False)

    keys = ["impression_uid", "news_code"]
    read_columns = list(dict.fromkeys([*keys, "prior_article_exposures", *columns]))
    features = (
        pl.scan_parquet(feature_files)
        .filter(pl.col("impression_uid").is_in(chosen))
        .select(read_columns).collect().to_pandas()
    )
    labels = (
        pl.scan_parquet(label_files)
        .filter(pl.col("impression_uid").is_in(chosen))
        .select(*keys, "clicked").collect().to_pandas()
    )
    train = features.merge(labels, on=keys, how="inner", validate="one_to_one")
    if not len(train) == len(features) == len(labels):
        raise ValueError("Training feature and label keys do not match")
    return train.sort_values(keys).reset_index(drop=True)

## 3.2 Fit Feature Preprocessing

Missing-value handling, scaling and one-hot encoding are fitted on the training sample
only and reused, unchanged, by every trial and by validation and test scoring.

In [ ]:
def build_preprocessor():
    """Median-impute and standardise numeric inputs; mode-impute and one-hot encode categoricals."""
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric, NUMERIC),
        ("categorical", categorical, CATEGORICAL),
    ])

In [ ]:
train = sample_train_impressions(TRAIN_IMPRESSIONS, FEATURES)
preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(train[FEATURES]).astype(np.float32, copy=False)
y_train = train["clicked"].to_numpy(np.float32)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")
print(f"Training impressions: {train['impression_uid'].nunique():,}")
print(f"Retained candidates: {len(train):,}; click rate: {y_train.mean():.3%}; input width: {X_train.shape[1]}")

## 3.3 Sample Within-Impression Preference Pairs

BPR compares a clicked candidate with an unclicked candidate from **the same impression**.
A non-click is an observed signal, not proof of dislike.

In [ ]:
def sample_pairs(train, *, popularity_alpha):
    """Pair each click with up to NEGATIVES_PER_POSITIVE unclicked candidates from its impression.

    Negatives are drawn without replacement with weight (1 + prior exposures) ** alpha:
    alpha = 0 is uniform and larger alpha favours popular articles. Impressions without
    an unclicked candidate contribute no pairs.
    """
    rng = np.random.default_rng(SEED)
    clicked = train["clicked"].to_numpy()
    popularity = train["prior_article_exposures"].to_numpy(dtype=float)

    positive_rows, negative_rows = [], []
    for rows in train.groupby("impression_uid", sort=False).indices.values():
        positives = rows[clicked[rows] == 1]
        negatives = rows[clicked[rows] == 0]
        if not len(negatives):
            continue
        # Work in log space so very popular articles cannot overflow the weights
        log_weights = popularity_alpha * np.log1p(popularity[negatives])
        weights = np.exp(log_weights - log_weights.max())
        probabilities = weights / weights.sum()
        for positive in positives:
            size = min(NEGATIVES_PER_POSITIVE, len(negatives))
            sampled = rng.choice(negatives, size=size, replace=False, p=probabilities)
            positive_rows.extend([positive] * len(sampled))
            negative_rows.extend(sampled.tolist())
    return np.asarray(positive_rows, dtype=np.int64), np.asarray(negative_rows, dtype=np.int64)

# 4. Model: `TabularMLP`

```text
Preprocessed tabular features
        → Dense 256 → ReLU → Dropout 0.20
        → Dense 64  → ReLU → Dropout 0.20
        → candidate logit  → sigmoid ranking score
```

In [ ]:
class TabularMLP(nn.Module):
    """Feed-forward network that maps one candidate's features to one logit."""

    def __init__(self, input_width, hidden_dims, dropout):
        super().__init__()
        layers, width = [], input_width
        for hidden in hidden_dims:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(dropout)])
            width = hidden
        self.network = nn.Sequential(*layers, nn.Linear(width, 1))

    def forward(self, features):
        # features: [batch, input_width] → logits: [batch]
        return self.network(features).squeeze(-1)

# 5. Training

## 5.1 Training Loop

- **BCE** iterates over shuffled candidate batches, with a positive-class weight derived
  from the training click rate
- **BPR** iterates over shuffled click/non-click pairs: $-\log\sigma(s^+ - s^-)$
- **Hybrid** draws one candidate batch and one pair batch per step, so repeated pair
  members do not distort the BCE candidate distribution

In [ ]:
def train_mlp(pairs, *, objective, hybrid_weight=0.5):
    """Train a fresh TabularMLP with the BCE, BPR or hybrid objective."""
    x = torch.from_numpy(X_train)
    y = torch.from_numpy(y_train)
    click_rate = y_train.mean()
    positive_weight = torch.tensor([(1 - click_rate) / click_rate], device=DEVICE)

    torch.manual_seed(SEED)
    model = TabularMLP(x.shape[1], HIDDEN_DIMS, DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    row_loader = DataLoader(
        TensorDataset(torch.arange(len(y))),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    pair_loader = None
    if objective != "bce":
        pair_loader = DataLoader(
            TensorDataset(torch.from_numpy(pairs[0]), torch.from_numpy(pairs[1])),
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=torch.Generator().manual_seed(SEED + 1),
        )

    for epoch in range(EPOCHS):
        model.train()
        pair_batches = cycle(pair_loader) if pair_loader is not None else None
        losses = []
        for (rows,) in row_loader:
            optimizer.zero_grad()

            if objective in ("bce", "hybrid"):
                logits = model(x[rows].to(DEVICE))
                bce = F.binary_cross_entropy_with_logits(logits, y[rows].to(DEVICE), pos_weight=positive_weight)
            if objective in ("bpr", "hybrid"):
                positive, negative = next(pair_batches)
                margin = model(x[positive].to(DEVICE)) - model(x[negative].to(DEVICE))
                bpr = -F.logsigmoid(margin).mean()

            if objective == "bce":
                loss = bce
            elif objective == "bpr":
                loss = bpr
            else:
                loss = (1 - hybrid_weight) * bce + hybrid_weight * bpr

            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        print(f"  epoch {epoch + 1}/{EPOCHS}: mean batch loss {np.mean(losses):.5f}")
    return model

## 5.2 Run One Trial

Each trial trains a new model, saves its weights and metadata, then scores and evaluates
every validation candidate with the shared evaluator.

In [ ]:
def mlp_predictor(model):
    """Wrap a trained MLP as a predict(frame) function for the evaluator."""
    def predict(frame):
        features = preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)
        model.eval()
        scores = []
        with torch.no_grad():
            for start in range(0, len(features), BATCH_SIZE):
                batch = torch.from_numpy(features[start:start + BATCH_SIZE]).to(DEVICE)
                scores.append(torch.sigmoid(model(batch)).cpu().numpy())
        return np.concatenate(scores)
    return predict


def run_trial(name, objective, *, popularity_alpha=0.0, hybrid_weight=None):
    """Train, save and validate one loss/sampling configuration."""
    pairs = sample_pairs(train, popularity_alpha=popularity_alpha)
    print(f"\n{name}: {objective}, popularity alpha={popularity_alpha}, pairs={len(pairs[0]):,}")
    model = train_mlp(pairs, objective=objective, hybrid_weight=hybrid_weight or 0.5)

    trial_dir = OUTPUT_DIR / name
    trial_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {"state_dict": model.state_dict(), "input_width": X_train.shape[1],
         "hidden_dims": HIDDEN_DIMS, "dropout": DROPOUT},
        trial_dir / "model.pt",
    )
    metadata = {
        "name": name, "objective": objective, "popularity_alpha": popularity_alpha,
        "hybrid_weight": hybrid_weight, "seed": SEED, "epochs": EPOCHS,
        "training_impressions": int(train["impression_uid"].nunique()),
        "training_rows": len(train), "pair_count": len(pairs[0]),
        "negatives_per_positive": NEGATIVES_PER_POSITIVE,
    }
    (trial_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

    result, _ = score_and_evaluate(
        mlp_predictor(model),
        split="validation",
        feature_run=RUN,
        output_dir=trial_dir,
        model_name=name,
        columns=FEATURES,
        results_dir=RESULTS,
    )
    return result.with_columns(
        pl.lit(objective).alias("objective"),
        pl.lit(float(popularity_alpha)).alias("popularity_alpha"),
        pl.lit(hybrid_weight, dtype=pl.Float64).alias("hybrid_weight"),
    )

# 6. Validation Experiments

## 6.1 BCE versus BPR Negative Sampling

Run BCE once, then BPR with uniform (`alpha = 0`) and popularity-weighted
(`alpha = 0.5, 1.0`) negatives. The best BPR sampler is chosen on validation.

In [ ]:
validation_results = [run_trial("mlp_bce", "bce")]
for alpha in POPULARITY_ALPHAS:
    validation_results.append(run_trial(f"mlp_bpr_pop_{alpha:.2f}", "bpr", popularity_alpha=alpha))

bpr_results = compare_results(validation_results, sort_by=SELECTION_METRIC).filter(pl.col("objective") == "bpr")
best_popularity_alpha = bpr_results["popularity_alpha"][0]
print(f"BPR sampling selected on validation: popularity alpha={best_popularity_alpha}")

## 6.2 Tune the Hybrid Loss Weight

All hybrid trials use the validation-selected BPR sampler.

In [ ]:
for weight in HYBRID_WEIGHTS:
    name = f"mlp_hybrid_bpr_{weight:.2f}_pop_{best_popularity_alpha:.2f}"
    validation_results.append(
        run_trial(name, "hybrid", popularity_alpha=best_popularity_alpha, hybrid_weight=weight)
    )

leaderboard = compare_results(validation_results, sort_by=SELECTION_METRIC)
leaderboard.write_csv(RESULTS / "reranking_mlp_loss_ablation_validation.csv")
final_model_name = leaderboard["model"][0]
print(f"Selected using validation {SELECTION_METRIC}: {final_model_name}")
display(leaderboard.select(
    "model", "objective", "popularity_alpha", "hybrid_weight",
    "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10",
))

# 7. Test Evaluation

Reload the validation-selected checkpoint and score every test candidate. Test results
are for final reporting only.

In [ ]:
checkpoint = torch.load(OUTPUT_DIR / final_model_name / "model.pt", map_location=DEVICE, weights_only=False)
final_model = TabularMLP(checkpoint["input_width"], checkpoint["hidden_dims"], checkpoint["dropout"]).to(DEVICE)
final_model.load_state_dict(checkpoint["state_dict"])

test_result, _ = score_and_evaluate(
    mlp_predictor(final_model),
    split="test",
    feature_run=RUN,
    output_dir=OUTPUT_DIR / final_model_name,
    model_name=final_model_name,
    columns=FEATURES,
    results_dir=RESULTS,
)
display(test_result.select("model", "split", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

# 8. Recommendations for a User

`rank_logged_slate` reorders the articles shown in a user's latest validation
impression using this notebook's saved scores. It does not retrieve new articles
from the catalogue and never reads click labels.

In [ ]:
example_user = int(validation_preview["user_code"].iloc[0])
rank_logged_slate(example_user, score_file=OUTPUT_DIR / final_model_name / "validation_scores.parquet", feature_run=RUN, top_n=5)